# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# %matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tpvalidator.workspace as workspace
import tpvalidator.analysis.snn as snn
import mplhep as hep

from rich import print


# Data

In [ ]:
import tpvalidator.datacatalogue as dctl

dataset_name = 'radbkg'
datasets = dctl.load('data/vd/1x8x14/3sig', dataset_name)
rad_ws=datasets[dataset_name]


In [ ]:
from tpvalidator.viz.tps import TrgPrimitivesPlotter

tpp = TrgPrimitivesPlotter(rad_ws)

In [ ]:

from matplotlib.colors import LogNorm

fig, ax = plt.subplots()
h = tpp.make_generator_counts_hist('bt_is_signal == 1')
norm = LogNorm(vmin=1, vmax=h.values().max())

hep.hist2dplot(h, ax=ax,  norm=norm)
ax.tick_params(axis='x', rotation=90)


In [ ]:
h[sum, :]

In [ ]:
fig = tpp.plot_generator_activity(cut='adc_peak>45', norm='rate')

In [ ]:
print(tpp.make_generator_rates_table( "adc_peak >45 & readout_plane_id==0"))
print(tpp.make_generator_rates_table( "adc_peak >45 & readout_plane_id==1"))
print(tpp.make_generator_rates_table( "adc_peak >45 & readout_plane_id==2"))

# TP sample validation
## Distribution of point of origin in the detector

In [ ]:
coll_tps = ws.tps.query('readout_view == 2 & sample_start >100 & sample_start < 8100 & bt_is_signal == 1 ')
coll_tps.extra_info.update({'readout_window': 8000})


In [ ]:
tps_by_gen = sorted([(n,df) for n,df in coll_tps.groupby('bt_generator_name')], reverse=True, key=lambda x: len(x[1]))


In [ ]:
counts = coll_tps.bt_generator_name.value_counts()
ax = counts.plot.bar()
ax.set_ylabel('counts')
ax.set_yscale('log')

In [ ]:
fig, ax = plt.subplots()

print(coll_tps.bt_generator_name.unique())
n_bins = coll_tps.bt_generator_name.nunique()
# coll_tps.bt_generator_name.hist(bins=n_bins)
coll_tps.bt_generator_name.hist(bins=n_bins)
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(ax.get_xticklabels(), rotation='vertical')
ax.set_ylabel('counts')
ax.set_yscale('log')

In [ ]:
fig, axes = plt.subplots()

ax=axes
alpha=0.01
col=None
cmap=None
coll_tps.plot.scatter(x='bt_primary_y', y='bt_primary_x', alpha=alpha, c=col, cmap=cmap, s=1, ax=ax)
ax.set_title('XY origin - Plane X [2]')

In [ ]:
top_by_gen = tps_by_gen[:10]


In [ ]:
# cmap = plt.get_cmap('Paired')
cmap = plt.get_cmap('tab10')
alpha=0.01
colors = [1,0,2,4,3,5,6,7,8,9]

from rich.table import Table

t = Table('Generator', 'Counts')
for n, df in top_by_gen:
    t.add_row(n, str(len(df)))

print(t)

tps_plot = top_by_gen[::-1]
col_sorted = colors[:len(tps_plot)][::-1]

fig, axes = plt.subplots(2,2, figsize=(8,8), layout='constrained')

ax = axes[0][0]
for i, (n, df) in enumerate(tps_plot):
    df.plot.scatter(x='bt_primary_y', y='bt_primary_x', alpha=alpha, color=cmap(col_sorted[i]), s=1, ax=ax)
ax.set_title("X-Y view")

ax = axes[0][1]
for i, (n, df) in enumerate(tps_plot):
    df.plot.scatter(x='bt_primary_z', y='bt_primary_x', alpha=alpha, color=cmap(col_sorted[i]), s=1, ax=ax)
ax.set_title("X-Y view")


ax = axes[1][0]
for i, (n, df) in enumerate(tps_plot):
    df.plot.scatter(x='bt_primary_y', y='bt_primary_z', alpha=alpha, color=cmap(col_sorted[i]), s=1, ax=ax)
ax.set_title("Y-Z view")

ax = axes[1][1]
ax.set_axis_off()
# fig.legend(loc='outside right upper')

from matplotlib.patches import Patch

# --- Proxy fill-style legend entries ---
legend_handles = [
    Patch(facecolor='tab:blue', alpha=0.5, label='Signal'),
    Patch(facecolor='tab:orange', alpha=0.5, label='Background'),
]


legend_handles = [ Patch(facecolor=cmap(colors[i]), label=n)  for i, (n, df) in enumerate(top_by_gen) ]

# --- Shared legend, outside subplots ---
fig.legend(
    handles=legend_handles,
    loc='lower right',
    # ncol=2,
    frameon=False,
    bbox_to_anchor=(1., 0.19)
    # loc='outside right upper'
)


In [ ]:

fig, ax = plt.subplots()

bins = np.linspace(-325, 325, 100)

colors = [1,0,2,4,3,5,6,7,8,9]


df = pd.concat([df for n, df in top_by_gen], axis=1)

ax.hist([df.bt_primary_x for n, df in top_by_gen], bins=bins, histtype='bar', stacked=True, color=[ cmap(colors[i]) for i, _ in enumerate(top_by_gen)], label=[n for n, df in top_by_gen])
ax.set_xlabel('Backtracked X')
ax.set_title('Origin of background TPs by generator')


# print(ax.get_ylim())
# ax.set_ylim(ymax=ax.get_ylim()[1]*10)
ax.set_ylim(1, ymax=10e6)
ax.set_yscale('log')
ax.grid()
ax.legend( prop={'size': 8})    

In [ ]:
all_tps = snn.TPSignalNoiseSelector(ws.tps)
alltp_ana = snn.TPSignalNoiseAnalyzer(all_tps)

In [ ]:
fig = alltp_ana.draw_tp_sig_origin_2d_dist()
fig.tight_layout()

In [ ]:
fig = alltp_ana.draw_tp_sig_drift_depth_dist()

In [ ]:
fig = alltp_ana.draw_tp_sig_drift_depth_dist(weight_by='adc_integral')

In [ ]:
fig = alltp_ana.draw_tp_sig_drift_depth_dist(weight_by='adc_peak', bins=1000)

# Dataset validation: TP distributions

### TP distribution in channel and time - one event with increasing peak ADC cuts

In [ ]:
x = snn.TPSignalNoiseAnalyzer(all_tps.query('adc_peak > 26'))
fig = x.draw_tp_event(10)
x = snn.TPSignalNoiseAnalyzer(all_tps.query('adc_peak > 36'))
fig = x.draw_tp_event(10)
x = snn.TPSignalNoiseAnalyzer(all_tps.query('adc_peak > 46'))
fig = x.draw_tp_event(10)
x = snn.TPSignalNoiseAnalyzer(all_tps.query('adc_peak > 56'))
fig = x.draw_tp_event(10)


### TP distribution in channel and time - all events

In [ ]:
fig = alltp_ana.draw_tp_start_sample_dist()

# Cleaning: removing regions with non-even backtracking efficiency

In [ ]:
tpw = snn.TPSignalNoiseSelector(ws.tps[(ws.tps.sample_start >100) & (ws.tps.sample_start <8100)])
tp_ana = snn.TPSignalNoiseAnalyzer(tpw)

In [ ]:
fig = tp_ana.draw_tp_start_sample_dist()


# adcpeak, time-over-threshold and SumADC distribution for Ar39 and noise

In [ ]:
fig = tp_ana.draw_tp_signal_noise_dist()
fig.tight_layout()

In [ ]:
fig = tp_ana.draw_variable_in_drift_grid('adc_peak', bin_size=10, sharex=True, sharey=True, figsize=(12,10))
fig.tight_layout()

In [ ]:
fig = tp_ana.draw_variable_in_drift_grid('adc_integral', bin_size=100, sharey=True, figsize=(12,10))
fig.tight_layout()

In [ ]:
fig = tp_ana.draw_variable_in_drift_grid('samples_over_threshold', bin_size=1, log=False, sharey=True, figsize=(12,10))
fig.tight_layout()

In [ ]:
fig = tp_ana.draw_variable_drift_stack('adc_peak', bin_size=5, n_x_bins=4, log=True, figsize=(5,4))
fig.tight_layout()

In [ ]:
fig = tp_ana.draw_variable_drift_stack('samples_over_threshold', bin_size=1, n_x_bins=4, log=False, figsize=(5,4))
fig.tight_layout()


In [ ]:
fig = tp_ana.draw_variable_drift_stack('adc_integral', bin_size=5, n_x_bins=4, log=True, figsize=(5,4))
fig.tight_layout()


In [ ]:
tot_cuts = [t for t in range(0,10,2)]

fig = tp_ana.draw_variable_cut_sequence('samples_over_threshold', tot_cuts, log=True, figsize=(10,10))


In [ ]:
cuts = [t for t in range(26, 50, 5)]

fig = tp_ana.draw_variable_cut_sequence('adc_peak', cuts, log=True, figsize=(10,10))
fig.tight_layout()

In [ ]:
cuts = [t for t in range(0, 500, 100)]

fig = tp_ana.draw_variable_cut_sequence('adc_integral', cuts, figsize=(10,10), log=True)
fig.tight_layout()

In [ ]:
import matplotlib as mpl
fig, axes = plt.subplots(2,2, figsize=(10,8))
norm=mpl.colors.LogNorm()

ides_clean = ws.ides.query('timestamp < 10000')


ax=axes[0][0]
ides_clean.x.hist(bins=100, ax=ax)
ax=axes[0][1]
ides_clean.x.hist(bins=100, weights=ides_clean.energy, ax=ax)
ax = axes[1][0]
h2d = ax.hist2d(ides_clean.x, ides_clean.timestamp, weights=ides_clean.energy, bins=(100, 100))
ax.set_ylabel('time')
ax.set_xlabel('depth')
cbar = fig.colorbar(h2d[3], ax=ax)
cbar.set_label('counts (energy weighted)')
fig.tight_layout()


In [ ]:
tp_bgd = ws.tps[ws.tps.samples_over_threshold >0]
# tp_bgd = ws.tps
# Count entries per generator and get top N culprits
N = 8
# tps_filtered = tp_bgd[(tp_bgd.readout_plane_id == 2)] # & (tp_bgd.samples_over_threshold > 0)]
tps_filtered = tp_bgd.query('bt_is_signal==1 & readout_plane_id == 2') # & (tp_bgd.samples_over_threshold > 0)]
generator_names_np = tps_filtered.bt_generator_name.to_numpy()
top_n = pd.Series(generator_names_np).value_counts().nlargest(N)
print(top_n)
top = pd.Series(generator_names_np).value_counts().nlargest(N).index
plt.yscale('log')
plt.xlabel('adc_integral')

bin_max = max([tps_filtered.loc[tps_filtered.bt_generator_name == top[i]]['adc_integral'].max() for i in range(0,len(top))])
bins=list(range(0, int(bin_max), 8))

for i in range(0,len(top)):
    plt.hist(tps_filtered.loc[tps_filtered.bt_generator_name == top[i], 'adc_integral'], bins=bins, histtype='step', label=f'{top[i]}')
plt.legend()
plt.show()
print(f"Rate per CRP\n{top_n/(100*8000*0.5e-6)/12} Hz")


In [ ]:
tp_bgd = ws.tps[ws.tps.samples_over_threshold >0]
# tp_bgd = ws.tps
# Count entries per generator and get top N culprits
N = 8
# tps_filtered = tp_bgd[(tp_bgd.readout_plane_id == 2)] # & (tp_bgd.samples_over_threshold > 0)]
tps_filtered = tp_bgd.query('bt_is_signal==1 & readout_plane_id == 2') # & (tp_bgd.samples_over_threshold > 0)]
generator_names_np = tps_filtered.bt_generator_name.to_numpy()
top_n = pd.Series(generator_names_np).value_counts().nlargest(N)
print(top_n)
top = pd.Series(generator_names_np).value_counts().nlargest(N).index
plt.yscale('log')
plt.xlabel('adc_integral')

bin_max = max([tps_filtered.loc[tps_filtered.bt_generator_name == top[i]]['samples_over_threshold'].max() for i in range(0,len(top))])
bins=list(range(bin_max))

for i in range(0,len(top)):
    plt.hist(tps_filtered.loc[tps_filtered.bt_generator_name == top[i], 'samples_over_threshold'], bins=bins, histtype='step', label=f'{top[i]}')
plt.legend()
plt.show()
print(f"Rate per CRP\n{top_n/(100*8000*0.5e-6)/12} Hz")


In [ ]:
len(ws.tps.query('bt_is_signal == 1 & readout_plane_id == 2'))/(100*8000*0.5e-6)/12*160/1e6

In [ ]:
from tpvalidator.utils import calculate_trg_obj_rates
from rich.table import Table


for i, df in tps_filtered.groupby('bt_generator_name'):
    print(i, calculate_trg_obj_rates(df, 8000))
    # break


pd.DataFrame([[g, calculate_trg_obj_rates(df, 8000)] for g, df in tps_filtered.groupby('bt_generator_name')])


In [ ]:
n_crp_fd = 160
n_crp_sim = 12
n_ev = len(ws.event_summary)
drift_time = 8000*16e-9*31.25

x = n_crp_fd/(n_crp_sim*drift_time*n_ev)


print(x, drift_time)

for i in range(0,len(top)):

    print(top[i], top_n.iloc[i], top_n.iloc[i]*x/1e6 )

In [ ]:
tpw.all.groupby('event').sample_start.max()-tpw.all.groupby('event').sample_start.min()

In [ ]:
tpw.p2.bt_x.hist()

In [ ]:
from tpvalidator.viz.backtracker import BackTrackerPlotter

In [ ]:
bt = BackTrackerPlotter(ws, 1)

In [ ]:
tps_ev1 = tpw.query('event==1')
display(tps_ev1.p2)

In [ ]:
tps_ev1.p2.query("adc_integral < 1000").iloc[:9]

In [ ]:
bt.plot_tps_vs_ides(bt.inspect_tps.query('adc_integral > 1000 & bt_is_signal == 1').iloc[:9], figsize=(12,12))
